In [10]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pydantic import BaseModel

load_dotenv()

model = init_chat_model(
    model="deepseek-flash"
)

class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户要求创建一个太空之都

# 指令
- 务必以 json 格式的内容结构返回，不要添加任何 markdown 格式

# 示例
- user：金星的首都是什么？
- assistant：
    {
        "name": "浮辉城（Vespera）"
        "location": "—— "悬浮在金星云层中的环形空中都市，以漂浮平台避开地表的高温与腐蚀性大气"
        "vibe": "清洁、高效"
        "economy": "城心的浮光塔收集高层云海中的风能，为整座城市供能。"
    }

"""

# agent = create_agent(
#     model=model,
#     system_prompt=system_prompt,
#     # response_format这种形式很多模型不支持，最好不要用这种模式
#     # response_format=CapitalInfo
# )

# response = agent.stream(
#     { "messages": [ { "role": "user", "content": "月球的首都是什么？" } ] },
#     stream_mode="messages"
# )

json_model = model.bind(response_format={"type": "json_object"})

buffer = ""
for chunk in json_model.stream(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "月球的首都是什么？"},
    ]
):
    if chunk.additional_kwargs.get("reasoning_content"):
        print(chunk.additional_kwargs["reasoning_content"], end="", flush=True)
    if chunk.content:
        buffer += chunk.content
        print(chunk.content, end="", flush=True)

print()
info = CapitalInfo.model_validate_json(buffer)   # 解析成 pydantic 对象
print(info)


{
  "name": "静海穹城（Selenopolis）",
  "location": "位于月球静海熔岩管与地表穹顶群之间，是月球联合殖民地的行政与交通中枢",
  "vibe": "冷峻、秩序井然、低重力下的高效",
  "economy": "依托熔岩管中的氦-3采掘、冰矿冶炼与地月轨道电梯中转贸易维持运转"
}我们需要回答用户中文问题。需要以json格式内容结构返回，不要markdown。身份科幻作家创建太空之都。用户问“月球的首都是什么？”需要虚构月球首都。必须符合 schema? 用户说 Response Format 必须严格 adhere {"type":"json_object"}。示例给了字段 name location vibe economy。需要输出 JSON 对象。不要 markdown。应该包含 name, location, vibe, economy 类似。用中文。需要创意。月球首都名称。可以叫“静海穹城（Selenopolis）”等。注意 JSON valid，不要注释。要返回纯 JSON。需要可能包含引号。字段值字符串。避免 trailing commas。可以四个字段。要回答“月球的首都是什么？” 可设定“静海都城（Mare Tranquillitatis capital）”。需要符合太空之都。以 json。只输出 JSON。不要 markdown。检查双引号。可以多字段。保持简洁。不要额外解释。用中文。JSON keys English. Values Chinese. 例如：
{
 "name": "静海穹城（Selenopolis）",
 "location": "位于月球静海熔岩管与地表穹顶群之间，是月球联合殖民地的行政与交通中枢",
 "vibe": "冷峻、秩序井然、低重力下的高效",
 "economy": "依托熔岩管中的氦-3采掘、冰矿冶炼与地月轨道电梯中转贸易维持运转。"
}
需要有效 JSON。这符合。注意“首都”可回答。可以包含 capital? 但 schema 没限制? Response Format only type json_object? 可能期望任意 JSON object。好。只输出。
name='静海穹城（Selenopolis）' location='位于月球静海熔岩管与地表穹